# Part 3 — Predictive Modeling with Spark ML
**Project:** Financial Forecasting Frontier: Distributed ML  
**Contribution:** Individual — Suman  
**Domain:** Banking marketing analytics  
**Data:** Official AlmaBetter bank.csv, 4,521 client/contact records, 17 columns.  
**Objective:** Explain distributed data processing and estimate term-deposit subscription. This is an educational sample, not a credit eligibility or fraud system.  
The supplied EDA/ML templates guide the summary → business problem → data → transformations → analysis/model → conclusion structure. Part-specific questions below follow the official question documents.


## Summary and business objective
Predict term-deposit subscription from information available before the current call. Compare a logistic model and a decision tree using training cross-validation, then report held-out performance. This is imbalanced classification: accuracy alone is misleading because predicting no for everyone is already about 88% accurate. Use ROC-AUC, PR-AUC, precision, recall, F1 and a confusion matrix. No financial decision is automated.

In [1]:
APP_NAME='Banking-Spark-ML'
import sys
sys.path.insert(0, '/workspace')
from banking.common import *
prepare_dirs()
spark = session(APP_NAME)
df = load_bank(spark).repartition(4).cache()
print('Spark version:', spark.version)
print('Master:', spark.sparkContext.master)
print('Data partitions:', df.rdd.getNumPartitions())


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/21 04:34:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3
Master: spark://spark-master:7077


Data partitions: 4


## Q1. Loading and initial exploration

In [2]:
df.printSchema();df.show(5,truncate=False)
df.groupBy('y').count().show()
print('Null cells:',sum(df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).first()))
print('Unknown values are valid source categories, not silently deleted.')


root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)



+---+-----------+--------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|job        |marital |education|default|balance|housing|loan|contact |day|month|duration|campaign|pdays|previous|poutcome|y  |
+---+-----------+--------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|26 |blue-collar|married |primary  |no     |5116   |yes    |no  |cellular|22 |jul  |246     |3       |-1   |0       |unknown |no |
|38 |management |single  |tertiary |no     |427    |yes    |no  |cellular|6  |may  |17      |3       |169  |2       |failure |no |
|33 |technician |single  |tertiary |no     |696    |no     |yes |cellular|5  |feb  |193     |1       |176  |8       |failure |no |
|50 |management |married |tertiary |no     |78     |yes    |no  |cellular|14 |jan  |218     |3       |161  |9       |other   |no |
|44 |admin.     |divorced|secondary|no     |432    |no     |no  |cellular|9  |feb  

+---+-----+
|  y|count|
+---+-----+
| no| 4000|
|yes|  521|
+---+-----+



Null cells: 0
Unknown values are valid source categories, not silently deleted.


## Q2. Preprocessing and outliers
Split before fitting imputers, indexers, encoders or models. Duration is excluded because it becomes known after a call. A fixed signed-log transform reduces the influence of extreme account balances while preserving negative balances; log1p reduces long campaign/previous-contact tails. These are deterministic transforms, so they learn no test statistics. Numerical medians are fitted within each cross-validation pipeline. Unknown categories remain explicit. pdays=-1 becomes a separate previous-contact flag plus nullable recency.

In [3]:
from banking.modeling import make_pipeline, evaluate, stable_split, transform_statement
df=features(df).filter(F.col('label').isNotNull())
train,test=stable_split(df)
train=train.repartition(4).cache();test=test.repartition(4).cache()
print('Training records:',train.count(),'Held-out records:',test.count())
print('Train labels:');train.groupBy('label').count().show()
print('Test labels:');test.groupBy('label').count().show()
print('Fixed transformations:',transform_statement())
assert 'duration' not in NUM and 'duration' not in CAT and 'y' not in CAT+NUM


26/10/21 04:36:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Training records: 3645 Held-out records: 876
Train labels:


+-----+-----+
|label|count|
+-----+-----+
|  0.0| 3221|
|  1.0|  424|
+-----+-----+

Test labels:


+-----+-----+
|label|count|
+-----+-----+
|  0.0|  779|
|  1.0|   97|
+-----+-----+

Fixed transformations: SELECT *, SIGN(balance)*LOG(1+ABS(balance)) AS balance_log,
    LOG(1+GREATEST(campaign,0)) AS campaign_log,
    LOG(1+GREATEST(previous,0)) AS previous_log FROM __THIS__


## Q3. Feature engineering and feature vector
StringIndexer and OneHotEncoder represent categorical features. Imputer fills numeric medians; VectorAssembler combines them; StandardScaler scales for logistic optimization. Every fitted preprocessing stage stays inside Spark ML Pipeline and CrossValidator. Preprocessing can therefore be saved and applied identically to streaming records.

In [4]:
from pyspark.ml.classification import LogisticRegression,DecisionTreeClassifier
lr=LogisticRegression(featuresCol='features',labelCol='label',maxIter=60)
dt=DecisionTreeClassifier(featuresCol='features',labelCol='label',seed=42)
lr_pipeline=make_pipeline(lr)
print('Pipeline stages:',[type(s).__name__ for s in lr_pipeline.getStages()])


Pipeline stages: ['SQLTransformer', 'Imputer', 'StringIndexer', 'OneHotEncoder', 'VectorAssembler', 'StandardScaler', 'LogisticRegression']


## Q4 and Q6. Classification, model selection and hyperparameter tuning
Three-fold CrossValidator uses a fixed seed and PR-AUC to focus on the rare positive class. The tree is an interpretable nonlinear comparison. Cross-validation scores select the winning family; test results never select a model or threshold. Default threshold 0.5 is retained and its limitations are reported.

In [5]:
from pyspark.ml.tuning import CrossValidator,ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
metric=BinaryClassificationEvaluator(labelCol='label',rawPredictionCol='probability',metricName='areaUnderPR')
candidates=[]
for name,model,grid in [
    ('logistic_regression',lr,ParamGridBuilder().addGrid(lr.regParam,[0.01,0.1]).addGrid(lr.elasticNetParam,[0.0,1.0]).build()),
    ('decision_tree',dt,ParamGridBuilder().addGrid(dt.maxDepth,[3,5]).addGrid(dt.minInstancesPerNode,[10]).build())]:
    cv=CrossValidator(estimator=make_pipeline(model),estimatorParamMaps=grid,evaluator=metric,numFolds=3,seed=42,parallelism=1)
    # Optional resume reuses only the already verified logistic CV result. Logistic raw-score and
    # probability rankings are monotonic equivalents. Fresh reproduction trains every candidate.
    if name=='logistic_regression' and os.environ.get('REUSE_LOGISTIC_CV')=='1':
        from pyspark.ml import PipelineModel
        cached=json.loads((ROOT/'artifacts/resume_lr_cv.json').read_text())
        print('Resuming verified logistic CV; positive-class probability ranking is equivalent:',cached)
        candidates.append((cached,PipelineModel.load(str(ROOT/'artifacts/resume_lr_model'))))
        continue
    fitted=cv.fit(train)
    scores=[float(x) for x in fitted.avgMetrics]
    best=max(range(len(scores)),key=scores.__getitem__)
    summary={'model':name,'cv_pr_auc':scores[best],'grid_scores':scores,'best_params':{p.name:v for p,v in grid[best].items()}}
    print(summary)
    candidates.append((summary,fitted.bestModel))
winner_summary,winner=max(candidates,key=lambda item:item[0]['cv_pr_auc'])
print('Selected using training CV only:',winner_summary)
winner.write().overwrite().save('/workspace/models/subscription_pipeline')
save_json('model_selection.json',{'candidates':[x[0] for x in candidates],'winner':winner_summary,'train_rows':train.count(),'test_rows':test.count(),'seed':42,'threshold':0.5,'excluded':['duration','y'],'split':'xxhash64 full source row, modulo 5; bucket 0 held out'})


Resuming verified logistic CV; positive-class probability ranking is equivalent: {'model': 'logistic_regression', 'cv_pr_auc': 0.36572544455690476, 'grid_scores': [0.3599725007123868, 0.3618471271560073, 0.36572544455690476, 0.2025250606048392], 'best_params': {'regParam': 0.1, 'elasticNetParam': 0.0}}


{'model': 'decision_tree', 'cv_pr_auc': 0.3990482012855083, 'grid_scores': [0.3990482012855083, 0.3623931886309035], 'best_params': {'maxDepth': 3, 'minInstancesPerNode': 10}}
Selected using training CV only: {'model': 'decision_tree', 'cv_pr_auc': 0.3990482012855083, 'grid_scores': [0.3990482012855083, 0.3623931886309035], 'best_params': {'maxDepth': 3, 'minInstancesPerNode': 10}}


## Q5. Held-out evaluation and baseline
PR-AUC is an area under the precision–recall curve, not accuracy. The no-skill PR baseline is test prevalence. A majority classifier has zero positive recall, despite high accuracy. Confidence is limited by the small test sample, random source sampling and absent customer identifiers; repeat clients cannot be grouped reliably. No results are interpreted as future default prediction.

In [6]:
metrics=evaluate(winner.transform(test))
positives=test.filter(F.col('label')==1).count();total=test.count()
baseline={'accuracy':1-positives/total,'positive_recall':0.0,'pr_auc_no_skill':positives/total}
print('Held-out metrics:',metrics)
print('Majority/no-skill baseline:',baseline)
save_json('model_metrics.json',{'selected_model':winner_summary['model'],'test':metrics,'baseline':baseline})
pred=winner.transform(test)
pred.select('label','prediction','probability').show(12,truncate=False)
test.select(SCHEMA.fieldNames()).coalesce(1).write.mode('overwrite').option('header',True).csv('/workspace/artifacts/heldout_source')


Held-out metrics: {'accuracy': 0.889269406392694, 'precision': 0.5, 'recall': 0.13402061855670103, 'f1': 0.21138211382113822, 'confusion': {'tn': 766, 'fp': 13, 'fn': 84, 'tp': 13}, 'roc_auc': 0.5797083228564244, 'pr_auc': 0.32642806091780463}
Majority/no-skill baseline: {'accuracy': 0.889269406392694, 'positive_recall': 0.0, 'pr_auc_no_skill': 0.11073059360730593}


+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
|0.0  |0.0       |[0.8636363636363636,0.13636363636363635]|
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
|0.0  |1.0       |[0.23943661971830985,0.7605633802816901]|
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
|0.0  |0.0       |[0.9056224899598394,0.09437751004016064]|
+-----+----------+----------------------------------------+
only showing top 12 rows



## Q7. Model interpretation
Coefficients refer to standardized features. They describe association within this model and cannot establish a causal marketing effect. For the tree, importances are impurity based and can favor features with many possible splits.

In [7]:
from banking.modeling import feature_importance
importance=feature_importance(winner,train)
for row in importance[:15]: print(row)
save_json('model_importance.json',importance)
train.unpersist();test.unpersist();df.unpersist();spark.stop()


{'feature': 'poutcome_vec_success', 'value': 0.6976868644145604}
{'feature': 'month_vec_oct', 'value': 0.15392332873980966}
{'feature': 'day_imputed', 'value': 0.08437372379446195}
{'feature': 'housing_vec_yes', 'value': 0.04118885316192775}
{'feature': 'balance_log_imputed', 'value': 0.022827229889240328}
{'feature': 'age_imputed', 'value': 0.0}
{'feature': 'campaign_log_imputed', 'value': 0.0}
{'feature': 'previous_log_imputed', 'value': 0.0}
{'feature': 'previously_contacted_imputed', 'value': 0.0}
{'feature': 'pdays_clean_imputed', 'value': 0.0}
{'feature': 'job_vec_blue-collar', 'value': 0.0}
{'feature': 'job_vec_management', 'value': 0.0}
{'feature': 'job_vec_technician', 'value': 0.0}
{'feature': 'job_vec_admin.', 'value': 0.0}
{'feature': 'job_vec_services', 'value': 0.0}


## Conclusion and practical limits
The saved Spark Pipeline includes preprocessing and can score structured streaming batches without refitting. Business use would require chronological/customer-group validation, probability calibration, fairness review, threshold selection on independent validation data and outcome monitoring. This educational model has not been approved for real banking use.